# uploadp.cptdaz.org (Front Door endpoint cptmtls-uploadp)


mTLS is enforced at the edge. The origin is a storage account without Private Link: Front Door requests a Microsoft Entra token with its user-assigned managed identity (origin authentication) and sends it as bearer token. The client needs only the client certificate, no SAS.

The storage account allows only Entra authentication (shared key disabled, anonymous access disabled).

Tools: bash, az, http (httpie), jq, azcopy, openssl.

## Design notes

* **Why a separate setup:** Origin authentication with a managed identity is not available for origins with Private Link. This storage account therefore has no Private Link: public network access is enabled, but only Microsoft Entra tokens are accepted (shared key and anonymous access are off). It carries the exclusion tag `SecurityControl=Ignore`, so the tenant policy does not reset the public network setting.
* **Origin authentication:** The origin group `og-uploadp-storage` uses the user-assigned identity `id-cptmtls-afd-st` with scope `https://storage.azure.com/.default`. The identity has Storage Blob Data Contributor on the account. Front Door adds the bearer token to every origin request.
* **mTLS at the edge:** Same CA and mode as `upload.cptdaz.org` (client certificate required and validated). The client needs only the certificate, no SAS.
* **Risk:** Every client with a valid certificate writes with the rights of the managed identity. Further use cases on this endpoint are in [upload-edge](upload-edge.ipynb) (path-scoped policy) and [upload-ja4](upload-ja4.ipynb) (WAF with JA4 fingerprints).
* **Separate identities:** The Key Vault access of the profile uses another identity (`id-cptmtls-afd-kv`), as recommended in the Front Door documentation.


In [ ]:
export SUB=09808f31-065f-4231-914d-776c2d6bbe34
export RG=rg-cptmtls
export DNS_SUB=ff0bb075-6c44-44ee-bb64-d46ce828c62f
export DNS_RG=domains
export DNS_ZONE=cptdaz.org
export PROFILE=afd-cptmtls-upload
export ENDPOINT=cptmtls-uploadp
export DOMAIN=uploadp-cptdaz-org
export HOST=uploadp.cptdaz.org
export STORAGE=stcptmtlsuploadpizhwq
export CONTAINER=upload
export IDENTITY=id-cptmtls-afd-st
export LAW=law-cptmtls
export PKI=$HOME/mtls-pki
export CERT=$PKI/client-valid.crt
export KEY=$PKI/client-valid.key
export BAD_CERT=$PKI/client-invalid.crt
export BAD_KEY=$PKI/client-invalid.key
export FILE=/tmp/uploadp-demo.txt
export BLOB=demo-$(date +%Y%m%d-%H%M%S).txt
export PAGER=cat
export AZCOPY_AUTO_LOGIN_TYPE=AZCLI
az account set --subscription $SUB
export LAW_ID=$(az monitor log-analytics workspace show -g $RG -n $LAW --query customerId -o tsv)
env | grep -E '^(SUB|RG|DNS_SUB|DNS_RG|DNS_ZONE|PROFILE|ENDPOINT|DOMAIN|HOST|STORAGE|CONTAINER|IDENTITY|LAW|PKI|CERT|KEY|BAD_CERT|BAD_KEY|FILE|BLOB|LAW_ID)=' | sort

## Azure account

In [ ]:
az account show --query "{name:name,id:id,user:user.name}" -o table

## Front Door configuration

In [ ]:
az afd profile show -g $RG --profile-name $PROFILE --query "{name:name,sku:sku.name,state:provisioningState}" -o table

In [ ]:
az rest --method get --url "https://management.azure.com/subscriptions/$SUB/resourceGroups/$RG/providers/Microsoft.Cdn/profiles/$PROFILE/afdEndpoints/$ENDPOINT?api-version=2026-07-01" | jq '{name, host: .properties.hostName, enforceMtls: .properties.enforceMtls, deployment: .properties.deploymentStatus}'

In [ ]:
az rest --method get --url "https://management.azure.com/subscriptions/$SUB/resourceGroups/$RG/providers/Microsoft.Cdn/profiles/$PROFILE/customDomains/$DOMAIN?api-version=2026-07-01" | jq '{host: .properties.hostName, validation: .properties.domainValidationState, deployment: .properties.deploymentStatus, mtls: .properties.mtlsSettings}'

In [ ]:
az afd route list -g $RG --profile-name $PROFILE --endpoint-name $ENDPOINT --query "[].{route:name,patterns:join(',',patternsToMatch),protocols:join(',',supportedProtocols),deployment:deploymentStatus}" -o table

## Origin authentication with the managed identity

In [ ]:
az rest --method get --url "https://management.azure.com/subscriptions/$SUB/resourceGroups/$RG/providers/Microsoft.Cdn/profiles/$PROFILE/originGroups/og-uploadp-storage?api-version=2026-07-01" | jq '.properties.authentication'

In [ ]:
az identity show -g $RG -n $IDENTITY --query "{name:name,principalId:principalId,clientId:clientId}" -o table

In [ ]:
az role assignment list --scope $(az storage account show -g $RG -n $STORAGE --query id -o tsv) --query "[].{role:roleDefinitionName,principalType:principalType,principalId:principalId}" -o table

In [ ]:
az storage account show -g $RG -n $STORAGE --query "{publicNetworkAccess:publicNetworkAccess,sharedKey:allowSharedKeyAccess,anonymousBlob:allowBlobPublicAccess,tags:tags}" -o json

## DNS records in the Azure DNS zone

In [ ]:
az network dns record-set cname show --subscription $DNS_SUB -g $DNS_RG -z $DNS_ZONE -n uploadp --query "{fqdn:fqdn,target:CNAMERecord.cname}" -o table

In [ ]:
az network dns record-set txt show --subscription $DNS_SUB -g $DNS_RG -z $DNS_ZONE -n _dnsauth.uploadp --query "{fqdn:fqdn,value:TXTRecords[0].value[0]}" -o table

## Client certificates (test PKI created with scripts/new-pki.sh)

In [ ]:
openssl verify -CAfile ../pki/ca.crt $CERT

In [ ]:
openssl verify -CAfile ../pki/ca.crt $BAD_CERT || true

In [ ]:
echo "hello from httpie (managed identity) $(date -u)" > $FILE
cat $FILE

## Storage account is not open without Entra token

An anonymous request directly to the storage account must be rejected.

In [ ]:
http --headers PUT https://$STORAGE.blob.core.windows.net/$CONTAINER/anonymous-$BLOB x-ms-blob-type:BlockBlob x-ms-version:2023-11-03 < $FILE

## Upload through Front Door

No client certificate: rejected at the edge.

In [ ]:
http --headers PUT https://$HOST/$CONTAINER/$BLOB x-ms-blob-type:BlockBlob x-ms-version:2023-11-03 < $FILE || true

Certificate from an untrusted CA: rejected at the edge.

In [ ]:
http --headers --cert=$BAD_CERT --cert-key=$BAD_KEY PUT https://$HOST/$CONTAINER/$BLOB x-ms-blob-type:BlockBlob x-ms-version:2023-11-03 < $FILE || true

Valid certificate, no SAS: Front Door adds the Entra token, the upload succeeds (`201 Created`).

In [ ]:
http --headers --cert=$CERT --cert-key=$KEY PUT https://$HOST/$CONTAINER/$BLOB x-ms-blob-type:BlockBlob x-ms-version:2023-11-03 < $FILE

## Verify the blob in the storage account

In [ ]:
az storage blob list --account-name $STORAGE --container-name $CONTAINER --auth-mode login -o json | jq '.[] | {name, size: .properties.contentLength, created: .properties.creationTime}'

In [ ]:
azcopy list "https://$STORAGE.blob.core.windows.net/$CONTAINER"

In [ ]:
azcopy copy "https://$STORAGE.blob.core.windows.net/$CONTAINER/$BLOB" /tmp/downloaded-$BLOB

In [ ]:
diff $FILE /tmp/downloaded-$BLOB && echo identical

## Download through Front Door with the certificate (Entra token added by Front Door)

In [ ]:
http --body --cert=$CERT --cert-key=$KEY GET https://$HOST/$CONTAINER/$BLOB x-ms-version:2023-11-03

## Direct upload with azcopy (bypasses Front Door)

In [ ]:
azcopy copy $FILE "https://$STORAGE.blob.core.windows.net/$CONTAINER/direct-$BLOB"

In [ ]:
azcopy list "https://$STORAGE.blob.core.windows.net/$CONTAINER"

## Clean up the demo blobs

In [ ]:
azcopy remove "https://$STORAGE.blob.core.windows.net/$CONTAINER/$BLOB"

In [ ]:
azcopy remove "https://$STORAGE.blob.core.windows.net/$CONTAINER/direct-$BLOB"

In [ ]:
az storage blob list --account-name $STORAGE --container-name $CONTAINER --auth-mode login --query "[].name" -o tsv

## Front Door access logs in Log Analytics

In [ ]:
az monitor log-analytics query -w $LAW_ID --analytics-query "AzureDiagnostics | where ResourceProvider == 'MICROSOFT.CDN' | where ResourceId contains toupper('$PROFILE') | summarize events=count() by Category" -o json | jq -c '.[]'

In [ ]:
az monitor log-analytics query -w $LAW_ID --analytics-query "AzureDiagnostics | where ResourceProvider == 'MICROSOFT.CDN' | where ResourceId contains toupper('$PROFILE') | top 5 by TimeGenerated desc | project TimeGenerated, Category, ResourceId" -o json | jq -c '.[]'